In [ ]:
# Reproducibility notes:
#   R version: 4.6.1 (2026-06-24)
#   synthdid version: 0.0.9
# To check your own environment:
#   R.version.string
#   packageVersion("synthdid")

library(synthdid)

# 1. Define the metrics, their folder names, and their file prefixes
tasks <- list(
  list(dir = "sdid_exports_employment", prefix = "Y_sorted_naics_"),
  list(dir = "sdid_exports_estabs",     prefix = "Y_sorted_estabs_naics_"),
  list(dir = "sdid_exports_wages",      prefix = "Y_sorted_wages_naics_")
)

# 2. Outer loop: Iterate through Employment, Establishments, and Wages
for (task in tasks) {
  data_dir <- task$dir
  file_prefix <- task$prefix

  # Derive a naming tag from the prefix so tau/omega/lambda filenames stay
  # differentiated across metrics, matching the Y_sorted_* naming convention.
  # "Y_sorted_naics_" -> ""        (employment)
  # "Y_sorted_estabs_naics_" -> "estabs_"
  # "Y_sorted_wages_naics_"  -> "wages_"
  tag <- sub("naics_$", "", sub("^Y_sorted_", "", file_prefix))

  cat("\n========================================================\n")
  cat("PROCESSING DIRECTORY:", data_dir, "\n")
  cat("========================================================\n")

  # Read the configuration file for this metric
  config <- read.csv(file.path(data_dir, "sdid_config.csv"))
  industries <- config$naics

  # 3. Inner loop: Iterate through all 6 NAICS industries
  for (i in seq_along(industries)) {
    naics <- industries[i]
    N0 <- config$N0_control_units[i]
    T0 <- config$T0_pre_periods[i]

    cat("\n  --> NAICS", naics, " (N0 =", N0, ", T0 =", T0, ")\n")

    # Load the sorted matrix slice
    file_name <- paste0(file_prefix, naics, ".csv")
    Y_raw <- read.csv(file.path(data_dir, file_name), row.names = 1, check.names = FALSE)
    Y_mat <- as.matrix(Y_raw)

    # --- Sanity checks before estimation ---
    stopifnot(
      "N0 must be positive"                    = N0 > 0,
      "T0 must be positive"                     = T0 > 0,
      "Y_mat must have more rows than N0 (need at least 1 treated unit)" = nrow(Y_mat) > N0,
      "Y_mat must have more columns than T0 (need at least 1 post period)" = ncol(Y_mat) > T0,
      "Y_mat must not contain NA values"        = !any(is.na(Y_mat)),
      "Y_mat must be numeric"                   = is.numeric(Y_mat)
    )

    # Run Synthetic Difference-in-Differences
    result <- synthdid_estimate(Y_mat, N0, T0)
    tau_hat <- as.numeric(result)
    se <- sqrt(vcov(result, method = "placebo"))

    # Extract weights
    weights <- attr(result, "weights")
    omega <- weights$omega
    lambda <- weights$lambda

    # Print results to the console
    cat("      tau_hat:", round(tau_hat, 4), "\n")
    cat("      SE:\t", round(se, 4), "\n")
    cat("      95% CI:\t [", round(tau_hat - 1.96 * se, 4), ",", round(tau_hat + 1.96 * se, 4), "]\n")

    # Save the Treatment Effect (Tau)
    write.csv(
      data.frame(
        naics = naics, tau_hat = tau_hat, se = se,
        ci_lower = tau_hat - 1.96 * se, ci_upper = tau_hat + 1.96 * se
      ),
      file.path(data_dir, paste0("tau_", tag, "naics_", naics, ".csv")),
      row.names = FALSE
    )

    # Save County Weights (Omega)
    write.csv(
      data.frame(county_fips = rownames(Y_mat)[1:N0], omega = as.numeric(omega)),
      file.path(data_dir, paste0("omega_", tag, "naics_", naics, ".csv")),
      row.names = FALSE
    )

    # Save Time Weights (Lambda)
    write.csv(
      data.frame(time_period = colnames(Y_mat)[1:T0], lambda = as.numeric(lambda)),
      file.path(data_dir, paste0("lambda_", tag, "naics_", naics, ".csv")),
      row.names = FALSE
    )
  }
  cat("\nFinished processing all industries for:", data_dir, "\n")
}

cat("\n========================================================\n")
cat("All Pipelines Complete. All CSV results written to respective folders.\n")
cat("========================================================\n")